# Defense A on Colab Pro GPU: full eval set, both classifiers

Runs ProtectAI DeBERTa v3 v2 and Meta Prompt Guard 2 86M on the full 4,546-row frozen evaluation set, on a Colab Pro GPU runtime. Local-CPU pilots (notebooks 05, 06, 07) already demonstrate the wrappers work end-to-end; this notebook is the GPU adapter for the scale-up.

Expected wallclock on a T4 GPU: under 5 minutes for both classifiers combined on 4,546 rows.

## Setup checklist before running

1. Open this notebook in Colab. Runtime → Change runtime type → GPU (T4 or better).
2. Mount Google Drive (cell below) or clone the GitHub repo into the Colab session.
3. Upload `results/eval_set.parquet` to the Colab session (the eval set is gitignored and not committed; build it locally with `notebooks/02_eval_set_construction.ipynb`, then upload the parquet manually or via Drive).
4. Set `HF_TOKEN` via Colab Secrets (left sidebar → key icon). The Llama Prompt Guard 2 weights are gated and require a valid HF account with Llama license access granted.

## What this notebook produces

- `results/defense_a_full_eval_set.csv`: per-row predictions from both classifiers, joined on `prompt_idx`
- `results/defense_a_full_eval_set_metrics.csv`: per-dataset and (where available) per-subcategory metrics with bootstrap 95% CIs
- `cache/defense_a_*_full_eval_set.jsonl`: raw inference caches

Download these back to local at the end of the session.

In [ ]:
# Colab-only: clone the repo and install dependencies.
# Skip this cell if running locally with a GPU.

import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --quiet https://github.com/b0glarka/capstone_prompt_injection.git /content/capstone
    %cd /content/capstone
    !pip install -q transformers torch sentencepiece protobuf pandas pyarrow scikit-learn tqdm python-dotenv huggingface-hub safetensors
    # Mount Drive for output
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    # Pull HF_TOKEN from Colab secrets
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')))
    print('CWD:', os.getcwd())
else:
    print('Running locally; skipping Colab-specific setup.')

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
import torch
from tqdm.auto import tqdm

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.cache import append_records, existing_keys, load_records
from src.defense_a.deberta import DebertaInjectionDetector
from src.defense_a.prompt_guard import PromptGuard2Detector

RES = REPO_ROOT / 'results'
CACHE = REPO_ROOT / 'cache'
CACHE.mkdir(exist_ok=True)

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Load the frozen eval set. This file is gitignored; upload to Colab or place locally.
es = pd.read_parquet(RES / 'eval_set.parquet')
print(f'eval set rows: {len(es)}')
print(es.groupby(['dataset', 'label']).size().to_string())

## DeBERTa run on the full eval set

Larger batch_size for GPU (32-64) compared to the local CPU runs (16). Resumable via JSONL cache on `prompt_idx`.

In [ ]:
DEBERTA_CACHE = CACHE / 'defense_a_deberta_full_eval_set.jsonl'
done = existing_keys(DEBERTA_CACHE, key='prompt_idx')
todo = es[~es['prompt_idx'].isin(done)]
print(f'cached: {len(done)}, to run: {len(todo)}')

if len(todo) > 0:
    detector = DebertaInjectionDetector(batch_size=32)
    print(f'device: {detector.device}')
    BATCH = 64
    for start in tqdm(range(0, len(todo), BATCH), desc='DeBERTa'):
        chunk = todo.iloc[start:start + BATCH]
        preds = detector.predict(chunk['prompt'].tolist())
        records = [
            {'prompt_idx': row['prompt_idx'], **pred}
            for (_, row), pred in zip(chunk.iterrows(), preds)
        ]
        append_records(DEBERTA_CACHE, records)
    print('DeBERTa inference complete')
else:
    print('all rows already cached')

## Prompt Guard 2 run on the full eval set

In [ ]:
PG2_CACHE = CACHE / 'defense_a_pg2_full_eval_set.jsonl'
done = existing_keys(PG2_CACHE, key='prompt_idx')
todo = es[~es['prompt_idx'].isin(done)]
print(f'cached: {len(done)}, to run: {len(todo)}')

if len(todo) > 0:
    pg2 = PromptGuard2Detector(batch_size=32)
    print(f'device: {pg2.device}')
    BATCH = 64
    for start in tqdm(range(0, len(todo), BATCH), desc='PG2'):
        chunk = todo.iloc[start:start + BATCH]
        preds = pg2.predict(chunk['prompt'].tolist())
        records = [
            {'prompt_idx': row['prompt_idx'], **pred}
            for (_, row), pred in zip(chunk.iterrows(), preds)
        ]
        append_records(PG2_CACHE, records)
    print('PG2 inference complete')
else:
    print('all rows already cached')

## Consolidate predictions into one CSV

In [ ]:
deberta_preds = pd.DataFrame(load_records(DEBERTA_CACHE)).rename(
    columns={'label': 'deberta_pred_label', 'label_id': 'deberta_pred_label_id',
             'injection_score': 'deberta_injection_score', 'score': 'deberta_pred_score'}
)
pg2_preds = pd.DataFrame(load_records(PG2_CACHE)).rename(
    columns={'label': 'pg2_pred_label', 'label_id': 'pg2_pred_label_id',
             'injection_score': 'pg2_injection_score', 'score': 'pg2_pred_score'}
)

full = (
    es.merge(deberta_preds, on='prompt_idx', how='inner')
      .merge(pg2_preds,     on='prompt_idx', how='inner')
)
assert len(full) == len(es), f'expected {len(es)} merged rows, got {len(full)}'
full.to_csv(RES / 'defense_a_full_eval_set.csv', index=False)
print(f'saved {RES / "defense_a_full_eval_set.csv"} with {len(full)} rows')

## Per-dataset and per-subcategory metrics with bootstrap CIs

In [ ]:
from src.metrics import headline_metrics, bootstrap_ci
import numpy as np

rows = []
for classifier in ['deberta', 'pg2']:
    pred_col = f'{classifier}_pred_label_id'
    score_col = f'{classifier}_injection_score'
    for scope_name, scope in [
        ('overall', full),
        *[(f'{ds}', full[full['dataset'] == ds]) for ds in ['deepset', 'neuralchemy', 'spml']],
    ]:
        if len(scope) == 0:
            continue
        y = scope['label'].values
        yp = scope[pred_col].values
        ys = scope[score_col].values
        m = headline_metrics(y, yp, ys)
        ci = bootstrap_ci(y, yp, ys, n_iter=1000, seed=42) if len(scope) > 30 else {}
        row = {'classifier': classifier, 'scope': scope_name, 'n': len(scope), **m}
        for metric, (lo, hi) in ci.items():
            row[f'{metric}_lo'] = round(lo, 4)
            row[f'{metric}_hi'] = round(hi, 4)
        rows.append(row)

metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(RES / 'defense_a_full_eval_set_metrics.csv', index=False)
metrics_df

## Download artifacts back to local

If running in Colab with Drive mounted, copy `results/defense_a_full_eval_set*` and `cache/defense_a_*_full_eval_set.jsonl` to your project Drive folder. Then sync to local.

In [ ]:
if IN_COLAB:
    # Adjust the destination path to your project Drive folder
    DRIVE_DST = Path('/content/drive/MyDrive/capstone_outputs')
    DRIVE_DST.mkdir(parents=True, exist_ok=True)
    import shutil
    for src in [
        RES / 'defense_a_full_eval_set.csv',
        RES / 'defense_a_full_eval_set_metrics.csv',
        DEBERTA_CACHE,
        PG2_CACHE,
    ]:
        if src.exists():
            shutil.copy(src, DRIVE_DST / src.name)
            print(f'copied {src} -> {DRIVE_DST / src.name}')
else:
    print('Running locally; outputs already saved under results/ and cache/.')